In [1]:
from sts.dio.equity import TickerDatabase as Ticker
import plotly.io as pio
import pandas as pd
import numpy as np
from sts.quant.consolidate import (
    get_consolidate_score_mm_multi_horizon,
)  # get_con_score_mm_multi_horizon
from sts.quant.breakout import get_breakout_mm_multi_horizon
from sts.quant.candle import Candle
from sts.quant.volatility import atr
import os
import traceback
import plotly.graph_objs as go
from sts.dio.symbols import sp500_symbol_table
from IPython.display import clear_output

pd.options.plotting.backend = "plotly"
pio.renderers.default = os.environ.get("PLOTLY_RENDERER", "notebook")

In [2]:
resample_rule = "1D"
rule_period_map = {"1D": "2y", "1W": "10y", "1M": "10y"}
period = rule_period_map[resample_rule]
consolidate_score_all_df = []
syms_with_error = []
for n, row in sp500_symbol_table.iterrows():
    try:
        sym = row["YahooSym"]
        print((n, sym), end=", ")
        ticker = Ticker()
        df = Candle(ticker.history(sym, period=period))
        if len(df) > 252:
            con_score_dict = get_consolidate_score_mm_multi_horizon(df.log(), window=20, buffer_threshold=0.1)
            breakout_score_dict = get_breakout_mm_multi_horizon(df.log(), window=20, buffer_threshold=0.1)
            volatilility = atr(df.log(), window=20)
            score = {}
            score["Sym"] = sym
            for key in con_score_dict.keys():
                ts = con_score_dict[key]
                ts = ts.rolling(window=3).mean()
                score[f"Con{key}"] = ts.iloc[-1]
            for key in breakout_score_dict.keys():
                ts = breakout_score_dict[key]
                ts = ts.rolling(window=3).mean()
                score[f"Break{key}"] = ts.iloc[-1]
            score["Vol"] = volatilility.iloc[-1] * np.sqrt(252)
            score["Sector"] = row["GICS Sector"]
            score["SubIndustry"] = row["GICS Sub-Industry"]
            consolidate_score_all_df.append(score)
            clear_output()
    except Exception as e:
        syms_with_error.append((sym, str(traceback.print_exc())))
consolidate_score_all_df = pd.DataFrame(consolidate_score_all_df)

In [3]:
consolidate_score_all_df = consolidate_score_all_df.rename(
    columns={
        "ConDailyScore": "CDScore",
        "ConWeeklyScore": "CWScore",
        "ConMonthlyScore": "CMScore",
        "BreakDailyScore": "BDScore",
        "BreakWeeklyScore": "BWScore",
        "BreakMonthlyScore": "BMScore",
    }
)

## kq Trending: improve, shift it a bit with long term moving average, can help capture various time horizon

In [37]:
sub_df = consolidate_score_all_df[
    (consolidate_score_all_df["BMScore"] > 0.8)
    & (consolidate_score_all_df["CWScore"] > 0.6)
    & (consolidate_score_all_df["BDScore"] > 0.6)
]
sub_df.sort_values("Sector", ascending=False).round(2)

,Sym,CDScore,CWScore,CMScore,BDScore,BWScore,BMScore,Vol,Sector,SubIndustry
460,VTR,0.00,1.00,0.00,0.67,0.00,1.0,0.27,Real Estate,Health Care REITs
247,PODD,0.00,0.67,0.00,1.00,0.67,1.0,0.40,Health Care,Health Care Equipment
385,DGX,0.00,1.00,0.00,1.00,0.00,1.0,0.30,Health Care,Health Care Services
20,ALL,0.00,1.00,0.33,1.00,0.00,1.0,0.32,Financials,Property & Casualty Insurance
62,WRB,0.00,1.00,0.00,1.00,0.00,1.0,0.25,Financials,Property & Casualty Insurance
107,CINF,0.67,0.67,0.33,1.00,0.33,1.0,0.31,Financials,Property & Casualty Insurance
222,HIG,0.00,0.67,0.00,1.00,0.33,1.0,0.25,Financials,Property & Casualty Insurance
298,MA,0.00,0.67,0.00,1.00,0.33,1.0,0.26,Financials,Transaction & Payment Processing Services
445,TRV,0.00,1.00,0.00,1.00,0.33,1.0,0.28,Financials,Property & Casualty Insurance
484,WTW,0.00,0.67,0.00,1.00,0.00,1.0,0.27,Financials,Insurance Brokers


## Curve Mean Reverting: down

In [29]:
sub_df = consolidate_score_all_df[
    (consolidate_score_all_df["CMScore"] > 0.6)
    & (consolidate_score_all_df["BWScore"] > 0.6)
    & (consolidate_score_all_df["BDScore"] < -0.6)
]
sub_df.sort_values("Sector", ascending=False).round(2)

,Sym,CDScore,CWScore,CMScore,BDScore,BWScore,BMScore,Vol,Sector,SubIndustry
357,PARA,0.0,0.0,1.0,-0.67,0.67,0.0,0.56,Communication Services,Movies & Entertainment


## Curve Mean Reverting: Up

In [32]:
sub_df = consolidate_score_all_df[
    (consolidate_score_all_df["CMScore"] > 0.6)
    & (consolidate_score_all_df["BWScore"] < -0.6)
    & (consolidate_score_all_df["BDScore"] > 0.1)
]
sub_df.sort_values("Sector", ascending=False).round(2)

,Sym,CDScore,CWScore,CMScore,BDScore,BWScore,BMScore,Vol,Sector,SubIndustry
126,CPRT,0.0,0.0,0.67,0.33,-0.67,0.0,0.30,Industrials,Diversified Support Services
374,PG,0.0,0.0,1.00,0.33,-0.67,0.0,0.23,Consumer Staples,Personal Care Products


### Strong Trending up

In [39]:
sub_df = consolidate_score_all_df[
    (consolidate_score_all_df["BMScore"] > 0.8)
    & (consolidate_score_all_df["BWScore"] > 0.5)
    & (consolidate_score_all_df["BDScore"] > 0.6)
]
sub_df.sort_values("Sector", ascending=False).round(2)

,Sym,CDScore,CWScore,CMScore,BDScore,BWScore,BMScore,Vol,Sector,SubIndustry
29,AEP,1.00,0.00,0.00,1.00,1.00,1.0,0.24,Utilities,Electric Utilities
181,EVRG,1.00,0.00,0.00,1.00,1.00,1.0,0.23,Utilities,Electric Utilities
93,CBRE,0.00,0.00,0.00,1.00,1.00,1.0,0.34,Real Estate,Real Estate Services
329,NEM,0.33,0.00,0.00,1.00,1.00,1.0,0.44,Materials,Gold
162,ECL,0.00,0.00,0.00,1.00,0.67,1.0,0.24,Materials,Specialty Chemicals
406,STX,0.00,0.00,0.00,1.00,1.00,1.0,0.53,Information Technology,"Technology Hardware, Storage & Peripherals"
37,APH,1.00,0.00,0.00,1.00,1.00,1.0,0.40,Information Technology,Electronic Components
46,ANET,1.00,0.00,0.00,1.00,1.00,1.0,0.61,Information Technology,Communications Equipment
127,GLW,1.00,0.00,0.00,1.00,1.00,1.0,0.35,Information Technology,Electronic Components
210,GEN,0.67,0.00,0.33,1.00,0.67,1.0,0.34,Information Technology,Application Software


## Monthly consolidate and weekly break up

In [26]:
sub_df = consolidate_score_all_df[
    (consolidate_score_all_df["CMScore"] > 0.6) & (consolidate_score_all_df["BWScore"] > 0.6)
]
sub_df.sort_values("Sector", ascending=False).round(2)

,Sym,CDScore,CWScore,CMScore,BDScore,BWScore,BMScore,Vol,Sector,SubIndustry
151,D,1.00,0.00,0.67,1.00,1.00,0.67,0.27,Utilities,Electric Utilities
469,VMC,0.67,0.00,0.67,1.00,1.00,0.33,0.31,Materials,Construction Materials
296,MLM,0.67,0.00,1.00,1.00,1.00,0.00,0.33,Materials,Construction Materials
284,LIN,0.00,0.33,1.00,1.00,0.67,0.00,0.21,Materials,Industrial Gases
318,MPWR,0.67,0.00,1.00,1.00,1.00,0.00,0.63,Information Technology,Semiconductors
278,LRCX,0.67,0.00,1.00,0.00,1.00,0.00,0.51,Information Technology,Semiconductor Materials & Equipment
39,ANSS,0.00,0.00,0.67,1.00,1.00,0.33,0.39,Information Technology,Application Software
26,AMD,0.67,0.00,1.00,0.00,1.00,0.00,0.68,Information Technology,Semiconductors
70,BA,1.00,0.00,1.00,0.00,1.00,0.00,0.40,Industrials,Aerospace & Defense
129,CSGP,0.67,0.00,1.00,0.00,1.00,0.00,0.38,Industrials,Research & Consulting Services


## Weekly consolidating and daily break up

In [25]:
sub_df = consolidate_score_all_df[
    (consolidate_score_all_df["CWScore"] > 0.8) & (consolidate_score_all_df["BDScore"] > 0.8)
]
sub_df.sort_values("Sector", ascending=False).round(2)

,Sym,CDScore,CWScore,CMScore,BDScore,BWScore,BMScore,Vol,Sector,SubIndustry
33,AWK,1.00,1.0,1.00,1.0,0.00,0.00,0.29,Utilities,Water Utilities
411,SPG,0.00,1.0,0.33,1.0,0.00,0.33,0.30,Real Estate,Retail REITs
192,FRT,0.00,1.0,1.00,1.0,0.00,0.00,0.30,Real Estate,Retail REITs
42,AAPL,0.67,1.0,0.33,1.0,0.00,0.00,0.34,Information Technology,"Technology Hardware, Storage & Peripherals"
236,HPQ,0.33,1.0,0.00,1.0,0.00,0.00,0.39,Information Technology,"Technology Hardware, Storage & Peripherals"
14,ALK,0.33,1.0,0.00,1.0,0.00,0.33,0.61,Industrials,Passenger Airlines
3,ABBV,0.00,1.0,0.00,1.0,0.00,0.33,0.33,Health Care,Pharmaceuticals
67,BIIB,0.00,1.0,0.00,1.0,0.00,-1.00,0.48,Health Care,Biotechnology
391,REGN,0.00,1.0,0.00,1.0,0.00,-1.00,0.48,Health Care,Biotechnology
385,DGX,0.00,1.0,0.00,1.0,0.00,1.00,0.30,Health Care,Health Care Services


## Long term Trending and short term  consolidates

In [40]:
sub_df = consolidate_score_all_df[
    (consolidate_score_all_df["CWScore"] > 0.6)
    & (consolidate_score_all_df["CDScore"] > 0.6)
    & (consolidate_score_all_df["BMScore"] > 0.6)
]
sub_df.sort_values("Sector", ascending=False).round(2)

,Sym,CDScore,CWScore,CMScore,BDScore,BWScore,BMScore,Vol,Sector,SubIndustry
97,CNP,1.00,1.00,0.00,0.00,0.00,1.00,0.26,Utilities,Multi-Utilities
114,CMS,1.00,1.00,0.00,0.00,0.00,1.00,0.23,Utilities,Multi-Utilities
183,EXC,1.00,1.00,0.00,0.67,0.00,0.67,0.26,Utilities,Multi-Utilities
368,PNW,1.00,1.00,0.33,0.00,0.00,1.00,0.26,Utilities,Multi-Utilities
448,TYL,1.00,1.00,0.67,0.00,0.00,0.67,0.35,Information Technology,Application Software
0,MMM,0.67,0.67,0.00,0.33,0.33,1.00,0.36,Industrials,Industrial Conglomerates
7,ADP,1.00,1.00,0.00,0.33,0.00,1.00,0.25,Industrials,Human Resource & Employment Services
393,RSG,1.00,1.00,0.00,0.00,0.00,1.00,0.24,Industrials,Environmental & Facilities Services
2,ABT,1.00,1.00,0.00,0.00,0.00,0.67,0.32,Health Care,Health Care Equipment
74,BSX,1.00,1.00,0.00,0.00,0.00,1.00,0.30,Health Care,Health Care Equipment


## Consolidate cross all regime

In [41]:
sub_df = consolidate_score_all_df[
    (consolidate_score_all_df["CWScore"] > 0.6)
    & (consolidate_score_all_df["CDScore"] > 0.6)
    & (consolidate_score_all_df["CMScore"] > 0.6)
]
sub_df.sort_values("Sector", ascending=False).round(2)

,Sym,CDScore,CWScore,CMScore,BDScore,BWScore,BMScore,Vol,Sector,SubIndustry
33,AWK,1.00,1.00,1.00,1.00,0.00,0.00,0.29,Utilities,Water Utilities
186,EXR,1.00,1.00,1.00,-0.67,0.00,0.00,0.33,Real Estate,Self-Storage REITs
271,KIM,0.67,0.67,0.67,0.33,0.00,0.00,0.31,Real Estate,Retail REITs
258,IRM,1.00,1.00,1.00,-0.67,0.00,0.00,0.39,Real Estate,Other Specialized REITs
73,BXP,1.00,1.00,1.00,0.00,0.00,0.00,0.42,Real Estate,Office REITs
380,PSA,0.67,0.67,1.00,0.33,0.00,0.00,0.29,Real Estate,Self-Storage REITs
157,DD,0.67,1.00,1.00,0.33,0.00,0.00,0.40,Materials,Specialty Chemicals
251,IP,1.00,1.00,0.67,0.00,0.00,0.00,0.44,Materials,Paper & Plastic Packaging Products & Materials
12,APD,0.67,1.00,0.67,0.33,0.00,0.00,0.29,Materials,Industrial Gases
421,STLD,0.67,1.00,1.00,0.00,0.00,0.00,0.45,Materials,Steel


## Long term down trend and start consolidate to break up

In [42]:
sub_df = consolidate_score_all_df[
    (consolidate_score_all_df["BMScore"] < -0.6)
    & (consolidate_score_all_df["CWScore"] > 0.6)
    & (consolidate_score_all_df["CDScore"] > 0.6)
]
sub_df.sort_values("Sector", ascending=False).round(2)

,Sym,CDScore,CWScore,CMScore,BDScore,BWScore,BMScore,Vol,Sector,SubIndustry
163,EIX,0.67,1.00,0.0,0.67,0.0,-0.67,0.42,Utilities,Electric Utilities
16,ARE,0.67,1.00,0.0,0.00,0.0,-1.00,0.47,Real Estate,Office REITs
481,WY,0.67,1.00,0.0,0.33,0.0,-0.67,0.37,Real Estate,Timber REITs
95,CE,0.67,1.00,0.0,-0.67,0.0,-1.00,0.85,Materials,Specialty Chemicals
199,FMC,1.00,1.00,0.0,0.00,0.0,-1.00,0.52,Materials,Fertilizers & Agricultural Chemicals
259,JBHT,0.67,1.00,0.0,0.00,0.0,-0.67,0.45,Industrials,Cargo Ground Transportation
125,COO,0.67,1.00,0.0,0.33,0.0,-0.67,0.39,Health Care,Health Care Supplies
142,XRAY,1.00,1.00,0.0,-0.33,0.0,-1.00,0.61,Health Care,Health Care Supplies
313,MRNA,1.00,1.00,0.0,-1.00,0.0,-1.00,0.75,Health Care,Biotechnology
352,OGN,1.00,1.00,0.0,0.00,0.0,-1.00,0.65,Health Care,Pharmaceuticals
